In [ ]:
import os, json
import pandas as pd
import numpy as np
from scipy import signal, interpolate
from scipy.io import loadmat, savemat
from tqdm import tqdm
import matplotlib.pyplot as plt

N_subj = 18
N=4
sec_dest = 124
tot_sec = 3600

In [ ]:
basePath = "/home/raffaelli/NonLinearity/NonLinearityData/iEEG"
csv_fname = f"iEEG_info.csv"
csv_path = os.path.join(basePath, csv_fname)
subj_data= pd.read_csv(csv_path, index_col=0)
def remove_spikes(EEG, fs_orig):
    fs_dest = 200
    border = int(0.05 * fs_dest) # remove 50 ms before and after the spike
    tollerance = round(0.12 * fs_dest)
    tot_samples_dest = round(EEG.shape[1] / fs_orig * fs_dest)

    sos = signal.butter(N, [10, 60], "bp", False, "sos", fs_orig)

    filtered = signal.sosfiltfilt(sos, EEG)
    resampled = signal.resample(filtered.T, tot_samples_dest)
    is_spike = np.zeros(resampled.shape[0], dtype=bool)
    for electrode in range(resampled.shape[1]):
        z = signal.hilbert(resampled[:, electrode])
        power = np.absolute(z)
        thresholds = []
        for window_start in range(0, resampled.shape[0] - fs_dest * 5, fs_dest):
            mu = np.mean(np.log(power[window_start : window_start + fs_dest * 5]))
            sigma = np.sqrt(
                np.sum(
                    (np.log(power[window_start : window_start + fs_dest * 5]) - mu) ** 2
                )
                / (fs_dest * 5 - 1)
            )
            mode_lognor = np.exp(mu - sigma**2)
            median_lognor = np.exp(mu)
            thresholds.append(3.65 * (mode_lognor + median_lognor))
        thresholds = [thresholds[0]] + thresholds + [thresholds[-1]]
        cs = interpolate.CubicSpline(
            np.concatenate(
                [
                    [0],
                    np.arange(0, resampled.shape[0] - fs_dest * 5, fs_dest)
                    + 2.5 * fs_dest,
                    [resampled.shape[0]],
                ]
            ),
            thresholds,
            bc_type="natural",
        )
        continuous_threshold = cs(np.arange(resampled.shape[0]))
        tmp = np.cumsum(continuous_threshold)
        tmp[fs_dest:] = tmp[fs_dest:] - tmp[:-fs_dest]
        mid_len = tmp[fs_dest - 1 :].shape[0]
        head = (resampled.shape[0] - mid_len) // 2
        tail = resampled.shape[0] - mid_len - head
        smooth_threshold = np.concatenate(
            [
                np.full(head, tmp[fs_dest - 1] / fs_dest),
                tmp[fs_dest - 1 :] / fs_dest,
                np.full(tail, tmp[-1] / fs_dest),
            ]
        )
        is_spike += power > smooth_threshold
    is_spike = is_spike.astype(int)
    is_spike[0] = 0
    is_spike[-1] = 0
    transitions = np.diff(is_spike)
    spike_begins = np.where(transitions == 1)[0]
    spike_ends = np.where(transitions == -1)[0]
    for n, begin in enumerate(spike_begins):
        near_spike = spike_ends[n - 1]
        if begin - tollerance < near_spike < begin:
            is_spike[near_spike:begin] = True
        is_spike[begin - border : begin] = True
        is_spike[spike_ends[n] : spike_ends[n] + border] = True
    indexes = (np.arange(EEG.shape[1]) / fs_orig * fs_dest).round().astype(int)
    indexes[indexes==is_spike.shape[0]]-=1
    return np.logical_not(is_spike[indexes])


bands = [[1.0, 4.0], [4.0, 8.0], [8.0, 12.0], [12.0, 30.0], [30.0, 44.0]]

num_electrodes = subj_data.Electrodes.min()
if not os.path.isfile(os.path.join(basePath, "usedElectrodes.json")):
    rs = np.random.RandomState(42)
    selected_electrodes = {}
    for i in tqdm(range(1, 19)):
        selected_electrodes[i] = np.sort(
            rs.choice(subj_data.loc[i, "Electrodes"], num_electrodes, False)
        ).tolist()

    with open(os.path.join(basePath, "usedElectrodes.json"), "w") as fp:
        json.dump(selected_electrodes, fp)
else:
    with open(os.path.join(basePath, "usedElectrodes.json"), "r") as fp:
        selected_electrodes = {int(k): v for k, v in json.load(fp).items()}


spike_mask = {}
for i in tqdm(range(1, N_subj + 1), desc=f"Generating IED masks"):
    mask_path = os.path.join(basePath, f"sub{i:02}_IEDmask.npy")
    if not os.path.isfile(mask_path):
        f_sample = subj_data.loc[i, "fs_Hz"]
        out_fname = f"iEEG_ID{i:02}.mat"
        in_path = os.path.join(basePath, out_fname)
        EEG = loadmat(in_path)["EEG"]

        start = (tot_sec - 5 * sec_dest) // 2 * f_sample
        stop = (tot_sec + 5 * sec_dest) // 2 * f_sample
        spike_mask[i] = remove_spikes(
            EEG[selected_electrodes[i], start:stop], f_sample
        )
        np.save(mask_path,spike_mask[i])
    else:
        spike_mask[i] = np.load(mask_path)


for j, band in enumerate(bands, start=1):  #
    fs_dest = int(band[1] * 2 * 1.125 + 0.5)
    tot_samp_dest = fs_dest * sec_dest
    new_dataset = np.empty([tot_samp_dest, num_electrodes, N_subj])
    for i in tqdm(range(1, N_subj + 1), desc=f"Band {j}"):
        f_sample = subj_data.loc[i, "fs_Hz"]
        tot_samp_masked = int(np.sum(spike_mask[i]) / f_sample * fs_dest)
        sos = signal.butter(N, band, "bp", False, "sos", f_sample)
        out_fname = f"iEEG_ID{i:02}.mat"
        in_path = os.path.join(basePath, out_fname)
        EEG = loadmat(in_path)["EEG"]

        start = (tot_sec - 5 * sec_dest) // 2 * f_sample
        stop = (tot_sec + 5 * sec_dest) // 2 * f_sample

        filtered = signal.sosfiltfilt(sos, EEG[selected_electrodes[i], start:stop][:, spike_mask[i]])
        # masked = filtered
        new_dataset[:, :, i - 1] = signal.resample(filtered.T, tot_samp_masked)[
            (tot_samp_masked - tot_samp_dest)
            // 2 : (tot_samp_masked - tot_samp_dest)
            // 2
            + tot_samp_dest,
            :,
        ]
    savemat(os.path.join(basePath, f"iEEG_despiked_band{j}.mat"), {"iEEG": new_dataset})

In [ ]:
i=1
f_sample = subj_data.loc[i, "fs_Hz"]
start = (tot_sec - 5 * sec_dest) // 2 * f_sample
stop = (tot_sec + 5 * sec_dest) // 2 * f_sample
out_fname = f"iEEG_ID{i:02}.mat"
in_path = os.path.join(basePath, out_fname)
EEG = loadmat(in_path)["EEG"][selected_electrodes[i], start:stop]
fs_dest = 200
tot_samples_dest = round(EEG.shape[1] / f_sample * fs_dest)

sos = signal.butter(N, [10, 60], "bp", False, "sos", f_sample)

filtered = signal.sosfiltfilt(sos, EEG)
resampled = signal.resample(filtered.T, tot_samples_dest)
z = signal.hilbert(resampled[:, :], axis=0)
power = np.absolute(z)
indexes = (np.arange(power.shape[0]) / fs_dest * f_sample).round().astype(int)
indexes[indexes==power.shape[0]]-=1


In [ ]:

x=np.arange(resampled.shape[0])/fs_dest
y=np.ma.masked_array(power, mask=np.logical_not(np.repeat(spike_mask[i][indexes], power.shape[1]).reshape([-1,power.shape[1]])))
for elec in [10,13]:
    thresholds = []
    for window_start in range(0, resampled.shape[0] - fs_dest * 5, fs_dest):
        mu = np.mean(np.log(power[window_start : window_start + fs_dest * 5,elec]))
        sigma = np.sqrt(
            np.sum(
                (np.log(power[window_start : window_start + fs_dest * 5,elec]) - mu) ** 2
            )
            / (fs_dest * 5 - 1)
        )
        mode_lognor = np.exp(mu - sigma**2)
        median_lognor = np.exp(mu)
        thresholds.append(3.65 * (mode_lognor + median_lognor))
    thresholds = [thresholds[0]] + thresholds + [thresholds[-1]]
    cs = interpolate.CubicSpline(
        np.concatenate(
            [
                [0],
                np.arange(0, resampled.shape[0] - fs_dest * 5, fs_dest)
                + 2.5 * fs_dest,
                [resampled.shape[0]],
            ]
        ),
        thresholds,
        bc_type="natural",
    )
    continuous_threshold = cs(np.arange(resampled.shape[0]))
    tmp = np.cumsum(continuous_threshold)
    tmp[fs_dest:] = tmp[fs_dest:] - tmp[:-fs_dest]
    mid_len = tmp[fs_dest - 1 :].shape[0]
    head = (resampled.shape[0] - mid_len) // 2
    tail = resampled.shape[0] - mid_len - head
    smooth_threshold = np.concatenate(
        [
            np.full(head, tmp[fs_dest - 1] / fs_dest),
            tmp[fs_dest - 1 :] / fs_dest,
            np.full(tail, tmp[-1] / fs_dest),
        ]
    )
    plt.figure(figsize=(20,5))
    plt.plot(x[:6000], power[:6000,elec], alpha=0.3)
    plt.plot(x[:6000], y[:6000,elec])
    plt.plot(x[:6000], smooth_threshold[:6000], label=f"Thres. el. {elec}")
    plt.xlim((0,30))
    plt.ylim(bottom=0.1)
    plt.yscale("log")
    plt.legend(loc=0)
    plt.show()